Optuna: defina os ranges no entorno dos valores a seguir

    tam_pop: int = [50,n_jobs],
    pe: float = [0,1]
    pm: float = [0,1],
    rho: float = [0,1],
    raio_cluster: float = [0,1]
    z_limiar: float = [1,2]
    votos_limiar: int = [2,10]
    step_init: float = 0.1,  - fixar este
    step_reduction: float = [0,1],
    min_step: float = 0.001, fiza este

In [ ]:
import os
import math
import time
import numpy as np
import random # Added: Import the random module
from typing import Tuple, List, Dict, Any, Callable

# =====================================================================
# 8. LEITURA DE INSTÂNCIAS E PREPARAÇÃO DOS DADOS
# =====================================================================

def ler_instancia(caminho_arquivo: str):
    with open(caminho_arquivo, 'r') as f:
        # Lê apenas linhas com conteúdo (limpa espaços e linhas em branco)
        linhas = [linha.strip() for linha in f if linha.strip()]

    # 1. Dimensões da instância (Linha 0: N jobs, M máquinas)
    dimensoes = list(map(int, linhas[0].split()))
    n, m = dimensoes[0], dimensoes[1]

    p_matrix = np.zeros((n, m), dtype=float)

    # 2. Leitura da Matriz de Tempos (Inicia em idx = 2 pulando a linha do cabeçalho "2")
    idx = 2
    for j in range(n):
        valores = list(map(int, linhas[idx].split()))
        # Filtra os índices ímpares (desconsidera os IDs das máquinas)
        p_matrix[j, :] = valores[1::2][:m]
        idx += 1

    # 3. Busca e Leitura dos Due Dates (prazos de entrega)
    due_dates = None
    duedate_encontrado = False

    # Procura a linha marcador "Duedate"
    while idx < len(linhas):
        if "duedate" in linhas[idx].lower():
            duedate_encontrado = True
            idx += 1
            break
        idx += 1

    # Extrai os N valores de due dates
    if duedate_encontrado:
        valores_dd = []
        while idx < len(linhas) and len(valores_dd) < n:
            nums = list(map(float, linhas[idx].split()))
            valores_dd.extend(nums)
            idx += 1

        due_dates = np.array(valores_dd[:n], dtype=float)

    return n, m, p_matrix, due_dates

def lerer_instancia(caminho_arquivo: str) -> Tuple[int, int, np.ndarray, np.ndarray]:
    """
    Lê o arquivo de instância (.txt) ignorando linhas em branco ou comentários.
    Retorna: (n_jobs, m_machines, p_matrix, due_dates)
    """
    with open(caminho_arquivo, 'r') as f:
        linhas = [l.strip() for l in f if l.strip() and not l.strip().startswith('#')]

    # Cabeçalho: N e M
    partes_cabecalho = linhas[0].split()
    n = int(partes_cabecalho[0])
    m = int(partes_cabecalho[1])

    # Matriz de Tempos de Processamento (N x M)
    p_matrix = np.zeros((n, m), dtype=int)
    idx = 2
    for j in range(n):
      valores = list(map(int, linhas[idx].split()))
      # Pega a partir do índice 1 pulando de 2 em 2 (1, 3, 5, 7, 9...)
      p_matrix[j, :] = valores[1::2][:m]
      idx += 1

    # Cabeçalho opcional de Duedate
    if idx < len(linhas) and linhas[idx].lower().startswith('due'):
        idx += 1
    idx += 2 # Pula a palavra 'Duedate'
    # Due Dates
    due_dates = []
    for j in range(n):
        due_dates.append(int(linhas[idx]))
        idx += 1

    return n, m, p_matrix, np.array(due_dates, dtype=int)


# =====================================================================
# 9. CÁLCULO DO LOWER BOUND (LIMITE INFERIOR - POTTS/MCMAHON-FLORIAN)
# =====================================================================
# def calcular_lower_bound(p_matrix: np.ndarray, due_dates: np.ndarray) -> int:
#     """
#     Calcula o Lower Bound global (LB) via relaxação de máquinas para T_max.
#     LB = max_j ( max(0, sum_m(p_{j,m}) - d_j) )
#     """
#     tempos_totais_jobs = np.sum(p_matrix, axis=1)
#     atrasos_minimos = np.maximum(0, tempos_totais_jobs - due_dates)
#     return int(np.max(atrasos_minimos))
def calcular_lower_bound_potts(p_matrix: np.ndarray, due_dates: np.ndarray) -> int:
    """
    Calcula o Lower Bound de McMahon-Florian / Potts via Relaxação por Máquina.
    Considera o congestionamento de todas as N tarefas em cada máquina m.
    """
    n, m = p_matrix.shape
    lb_por_maquina = []

    for maq in range(m):
        # Heads (r_j): tempo mínimo de chegada do job j até a máquina 'maq'
        r = np.sum(p_matrix[:, :maq], axis=1) if maq > 0 else np.zeros(n)

        # Tails (q_j): tempo mínimo de saída do job j após a máquina 'maq'
        q = np.sum(p_matrix[:, maq + 1:], axis=1) if maq < m - 1 else np.zeros(n)

        # Prazos de entrega ajustados para a máquina 'maq'
        due_dates_modificados = due_dates - q

        # Carga total acumulada por todas as N tarefas na máquina 'maq'
        carga_total_maquina = np.sum(p_matrix[:, maq])

        # Liberação mínima e cauda mínima entre todos os jobs
        min_r = np.min(r)

        # Estimativa de conclusão mínima na máquina m
        # LB na máquina m = min_r + carga_total - np.max(d'_j)
        lb_m = max(0, min_r + carga_total_maquina - np.max(due_dates_modificados))
        lb_por_maquina.append(lb_m)

    # Job-based bound tradicional (para garantia)
    lb_jobs = np.max(np.maximum(0, np.sum(p_matrix, axis=1) - due_dates))

    # O LB global é o máximo entre todos os gargalos por máquina e por job
    return int(max(np.max(lb_por_maquina), lb_jobs))


# =====================================================================
# 4. DECODIFICADOR TWO-PHASE (PFS / NPFS) - REFATORADO
# =====================================================================
def decodificar_two_phase(
    cromossomo: np.ndarray,
    n: int,
    m: int,
    p_matrix: np.ndarray,
    due_dates: np.ndarray = None,
    modo: str = "PFS",
    obj: str = "SPAM"
) -> float:
    """
    Decodifica o cromossomo contínuo no ambiente Wait Allowed (Flow Shop).

    Parâmetros:
      - cromossomo: Vetor contínuo de Random Keys.
                    Modo PFS  -> Tamanho N (Replicado para as M máquinas).
                    Modo NPFS -> Tamanho N * M (ou matriz M x N).
      - n: Número de tarefas (jobs).
      - m: Número de máquinas.
      - p_matrix: Matriz de tempos de processamento (N x M).
      - due_dates: Vetor de prazos de entrega (Tamanho N). Opcional se obj='SPAM'.
      - modo: 'PFS' (mesma sequência em todas as máquinas) ou 'NPFS' (sequências distintas).
      - obj: 'SPAM' / 'SPAN' / 'CMAX' (Makespan) ou 'LMAX' / 'TMAX' (Atraso Máximo).

    Retorna:
      - float: Valor da função objetivo calculada.
    """
    modo_upper = modo.upper().strip()
    obj_upper = obj.upper().strip()

    # 1. Obtenção da ordem de processamento por máquina
    if modo_upper == "PFS":
        # Ordena apenas os N primeiros genes para obter a sequência única
        seq_unica = np.argsort(cromossomo[:n])
        ordens_maquinas = [seq_unica for _ in range(m)]

    elif modo_upper == "NPFS":
        # Se for um vetor unidimensional de N*M, remodela para matriz M x N
        if cromossomo.ndim == 1:
            chaves_maquinas = cromossomo[:n * m].reshape((m, n))
        else:
            chaves_maquinas = cromossomo

        ordens_maquinas = [np.argsort(chaves_maquinas[maq, :]) for maq in range(m)]

    else:
        raise ValueError(f"Modo '{modo}' inválido. Utilize 'PFS' ou 'NPFS'.")

    # 2. Simulação do Chão de Fábrica (Wait Allowed)
    completion_time = np.zeros((n, m), dtype=float)

    # Máquina 0 (Primeiro Estágio)
    tempo_maq = 0.0
    for job in ordens_maquinas[0]:
        tempo_maq += p_matrix[job, 0]
        completion_time[job, 0] = tempo_maq

    # Máquinas Subsequentes (1 até M - 1)
    for maq in range(1, m):
        tempo_maq = 0.0
        for job in ordens_maquinas[maq]:
            pronto_job = completion_time[job, maq - 1]
            inicio = max(tempo_maq, pronto_job)
            completion_time[job, maq] = inicio + p_matrix[job, maq]
            tempo_maq = completion_time[job, maq]

    # 3. Retorno do Objetivo
    if obj_upper in ["SPAM", "SPAN", "CMAX", "MAKESPAN"]:
        return float(np.max(completion_time[:, m - 1]))

    elif obj_upper in ["LMAX", "TMAX"]:
        if due_dates is None:
            raise ValueError("O parâmetro 'due_dates' é obrigatório quando obj='LMAX'.")
        atrasos = np.maximum(0.0, completion_time[:, m - 1] - due_dates)
        return float(np.max(atrasos))

    else:
        raise ValueError(f"Objetivo '{obj}' inválido. Escolha 'SPAM' (Makespan) ou 'LMAX' (Atraso Máximo).")



# =====================================================================
# 5. BUSCA LOCAL HOOKE & JEEVES ADAPTADA
# =====================================================================
def hooke_jeeves_local_search(
    cromossomo_init: np.ndarray,
    avaliar_func: Callable[[np.ndarray], float],
    step_init: float = 0.1,
    step_reduction: float = 0.5,
    min_step: float = 0.001,
    max_evals: int = 500
) -> Tuple[np.ndarray, float, int]:
    """
    Hooke & Jeeves operando no espaço contínuo [0, 1]^D com restrição de clipping.
    """
    dim = len(cromossomo_init)
    best_x = np.clip(cromossomo_init.copy(), 0.0, 1.0)
    best_fit = avaliar_func(best_x)
    evals = 1

    x_base = best_x.copy()
    step = step_init

    while step > min_step and evals < max_evals:
        x_novo = x_base.copy()
        melhorou = False

        # Movimento Exploratório
        for i in range(dim):
            for sinal in [+1, -1]:
                if evals >= max_evals:
                    break
                x_teste = x_novo.copy()
                # Clipping Crítico
                x_teste[i] = np.clip(x_teste[i] + sinal * step, 0.0, 1.0)
                fit_teste = avaliar_func(x_teste)
                evals += 1

                if fit_teste < best_fit:
                    best_fit = fit_teste
                    x_novo = x_teste.copy()
                    melhorou = True
                    break

        if melhorou:
            x_base = x_novo.copy()
        else:
            step *= step_reduction

    return (best_x if best_fit < avaliar_func(best_x) else x_novo), best_fit, evals


# =====================================================================
# 4. CLASSE DE SUPORTE PARA CLUSTER STATEFUL
# =====================================================================
class ClusterStateful:
    def __init__(self, centro: np.ndarray, fitness: float):
        self.centro = centro.copy()
        self.votos = 1
        self.fitness_centro = fitness


# =====================================================================
# 2, 3, 6, 7. ALGORITMO PRINCIPAL (2P-BRKGA-SCS)
# =====================================================================
def run_brkga_scs(
    n: int,
    m: int,
    p_matrix: np.ndarray,
    due_dates: np.ndarray,
    lb: int,
    tam_pop: int = 100,
    pe: float = 0.2,   ## 0.15
    pm: float = 0.15,  ## 0.25
    rho: float = 0.7,  ## 0.85
    raio_cluster: float = 0.5,
    z_limiar: float = 1.5, ### 1.645
    votos_limiar: int = 5,
    step_init: float = 0.1,
    step_reduction: float = 0.5,
    min_step: float = 0.001,
    max_generations: int = 200,
    max_evals_total: int = 20000,
    seed: int = 42
) -> Dict[str, Any]:
    """
    Executa o 2P-BRKGA-SCS com Cross-Space Knowledge Transfer e Duplo Best Tracking.
    """
    np.random.seed(int(time.time())) # Corrected: Cast time.time() to int
    evals_count = 0

    # 2. Wrappers de Avaliação
    def avaliar_pfs(sol: np.ndarray) -> float:
        nonlocal evals_count
        evals_count += 1
        return decodificar_two_phase(sol, n, m, p_matrix, due_dates, modo="PFS",obj="SPAM")

    def avaliar_npfs(sol: np.ndarray) -> float:
        nonlocal evals_count
        evals_count += 1
        return decodificar_two_phase(sol, n, m, p_matrix, due_dates, modo="NPFS",obj="SPAM")  ### obj="SPAM"

    # Duplo Best Rastreamento
    best_global_pfs = None
    best_fitness_pfs = float('inf')

    best_global_npfs = None
    best_fitness_npfs = float('inf')

    # 3. Inicialização BRKGA (Espaço Reduzido D = N)
    dim_pfs = n
    n_elite = max(1, int(tam_pop * pe))
    n_mutants = max(1, int(tam_pop * pm))
    n_crossover = tam_pop - n_elite - n_mutants

    populacao = np.random.uniform(0.0, 1.0, size=(tam_pop, dim_pfs))
    fitness_pfs = np.array([avaliar_pfs(ind) for ind in populacao])

    clusters: List[ClusterStateful] = []

    # Loop Evolutivo
    for gen in range(max_generations):
        if evals_count >= max_evals_total or best_fitness_npfs <= lb:
            break

        # Ordenação por Fitness PFS
        idx_ordenados = np.argsort(fitness_pfs)
        populacao = populacao[idx_ordenados]
        fitness_pfs = fitness_pfs[idx_ordenados]

        # Rastreamento Best PFS
        if fitness_pfs[0] < best_fitness_pfs:
            best_fitness_pfs = fitness_pfs[0]
            best_global_pfs = populacao[0].copy()

        # Early Stopping ACMOB
        if best_fitness_pfs <= lb:
            best_fitness_npfs = min(best_fitness_npfs, best_fitness_pfs)
            break

        # 4. MÓDULO DE CLUSTERING (Atualização com Elite)
        for i in range(n_elite):
            elite_ind = populacao[i]
            elite_fit = fitness_pfs[i]

            min_dist = float('inf')
            nearest_cluster = None

            for c in clusters:
                dist = np.linalg.norm(c.centro - elite_ind)
                if dist < min_dist:
                    min_dist = dist
                    nearest_cluster = c
    # 1. Cálculo do Raio Dinâmico da Literatura
            raio_cluster = 1.0 / (2 * (clusters ** (1 / n)))
            if nearest_cluster is not None and min_dist < raio_cluster:
                nearest_cluster.votos += 1
                # Substitui centro se o elite for superior (100% peso ao melhor)
                if elite_fit < nearest_cluster.fitness_centro:
                    nearest_cluster.centro = elite_ind.copy()
                    nearest_cluster.fitness_centro = elite_fit
            else:
                clusters.append(ClusterStateful(centro=elite_ind, fitness=elite_fit))

        # 6. CROSS-SPACE KNOWLEDGE TRANSFER & SCS
        media_pop = np.mean(fitness_pfs)
        std_pop = np.std(fitness_pfs) + 1e-9

        for c in clusters:
            if evals_count >= max_evals_total or best_fitness_npfs <= lb:
                break

            z_score = (c.fitness_centro - media_pop) / std_pop

            # Gatilho de Ativação do SCS
            if (z_score <= -z_limiar) or (c.votos >= votos_limiar):
                # Passo 1: Projeção de Conhecimento (PFS -> NPFS via np.tile)
                cromossomo_expandido = np.tile(c.centro, m)

                # Passo 2: Explotação via Hooke & Jeeves no Espaço NPFS
                budget_restante = max_evals_total - evals_count
                sol_refinada, fit_refinado, evals_hj = hooke_jeeves_local_search(
                    cromossomo_expandido,
                    avaliar_npfs,
                    step_init=step_init,
                    step_reduction=step_reduction,
                    min_step=min_step,
                    max_evals=min(500, budget_restante)
                )

                # Passo 3: Atualização do NPFS Best
                if fit_refinado < best_fitness_npfs:
                    best_fitness_npfs = fit_refinado
                    best_global_npfs = sol_refinada.copy()
                    print(f"New Best NPFS: {best_fitness_npfs}")

                # Passo 4: Reset do Estado do Cluster
                c.votos = 0

        if best_fitness_npfs <= lb:
            break

        # 3. Evolução da População (BRKGA Crossover & Mutantes)
        nova_pop = np.zeros_like(populacao)
        nova_pop[:n_elite] = populacao[:n_elite]
        nova_pop[n_elite:n_elite + n_mutants] = np.random.uniform(0.0, 1.0, size=(n_mutants, dim_pfs))

        for i in range(n_crossover):
            pai_e = populacao[np.random.randint(0, n_elite)]
            pai_ne = populacao[np.random.randint(n_elite, tam_pop)]
            mascara = np.random.rand(dim_pfs) < rho
            nova_pop[n_elite + n_mutants + i] = np.where(mascara, pai_e, pai_ne)

        populacao = nova_pop
        fitness_pfs = np.array([avaliar_pfs(ind) for ind in populacao])
    #fitex = decodificar_two_phase(cromossomo=pai_e, n=16, m=5,p_matrix=p_matrix, due_dates=None, modo="PFS", obj="SPAM")
    #print(f"CHECKOUT: {fitex}")
    return {
        "best_global_pfs": best_global_pfs,
        "best_fitness_pfs": best_fitness_pfs,
        "best_global_npfs": best_global_npfs,
        "best_fitness_npfs": best_fitness_npfs,
        "total_evaluations": evals_count,
        "lower_bound": lb
    }


# =====================================================================
# BLOCO DE EXECUÇÃO DE EXEMPLO / TESTE SANIDADE
# ==========================Raio===========================================
# /content/drive/MyDrive/instances/Small/I_2_16_5_1.txt
# I_2_10_4_4
if __name__ == "__main__":
    # Teste rápido simulado sem depender de arquivo externo
    n_test,m_test,p_matrix_test,due_dates_test = ler_instancia("/content/drive/MyDrive/instances/Large/Ta090_2.txt")
    # n_test, m_test = 5, 3
    # p_matrix_test = np.array([
    #     [10, 15, 20],
    #     [5,  10, 25],
    #     [30, 12, 8],
    #     [15, 20, 10],
    #     [8,  12, 14]
    # ], dtype=int)
    # due_dates_test = np.array([40, 50, 60, 45, 35], dtype=int)

    lb_test = calcular_lower_bound_potts(p_matrix_test, due_dates_test)
    print(f"Lower Bound Calculado (McMahon-Florian/Potts): {lb_test}")

    resultado = run_brkga_scs(
        n=n_test,
        m=m_test,
        p_matrix=p_matrix_test,
        due_dates=due_dates_test,
        lb=lb_test,
        max_generations=9999999,
        max_evals_total=100*20*1000,   ## 60nm ms 200nm
        seed=time.time()
    )


    print("\n================ RESULTADOS DOS DOIS ESPAÇOS ================")
    print(f"PFS Best Fitness (BRKGA Macro) : {resultado['best_fitness_pfs']}")
    print(f"NPFS Best Fitness (SCS Micro)  : {resultado['best_fitness_npfs']}")
    print(f"Avaliações Totais Utilizadas   : {resultado['total_evaluations']}")
    print("=============================================================")

Lower Bound Calculado (McMahon-Florian/Potts): 2249


TypeError: unsupported operand type(s) for ** or pow(): 'list' and 'float'

Olá! Estou progredindo nos testes e avançando para as instâncias grandes de Taillard. Nesse ponto o GAP em relação aos upper bounds de literatura estão oscilando entre 1.5% e 3.5%, o que não é tão ruim. Revisitando os hyper-parâmetros, percebi que não aproveitamos nada dos experimentos optuna. Essé um ponto que precisamos revisitar. Sintonizar para instâncias 100x10, a 200x10 (médias) pelo menos. Mas antes disso, percebi que algo se perdeu na última versão do Clustering Search. Vou enumerar as duas características principais: 1) raio adaptativo usando a Distância de Chebyshev 2) A Mecânica da Memória de Estado (Stateful CS) com mecanismo de Cooldown geométrico e Dinâmico. Temos com recuperar isso?

In [ ]:
import numpy as np

class StatefulClusteringSearch:
    def __init__(self, num_clusters, dimensao, delta_start=0.1, epsilon=1e-4):
        self.K = num_clusters
        self.D = dimensao
        self.delta_start = delta_start
        self.epsilon = epsilon

        # Estruturas de dados ultra-leves (apenas K elementos)
        self.centros = np.random.rand(num_clusters, dimensao)
        self.deltas = np.full(num_clusters, delta_start)      # Guarda o passo atual do H&J
        self.ultimos_fitness = np.full(num_clusters, np.inf) # Histórico de melhoria

    def executar_ciclo(self, novas_solucoes, fitness_novas_solucoes, engine, budget_hj=500):
        votos = np.zeros(self.K)
        raio = 1.0 / (2 * (self.K ** (1 / self.D)))

        # 1. Agrupamento e Assimilação
        for S, fit in zip(novas_solucoes, fitness_novas_solucoes):
            distancias = np.max(np.abs(self.centros - S), axis=1)
            cluster_idx = np.argmin(distancias)

            if distancias[cluster_idx] <= raio:
                # O centro se move por arrasto aritmético
                self.centros[cluster_idx] = 0.8 * self.centros[cluster_idx] + 0.2 * S
                votos[cluster_idx] += 1

                # Se o centro se moveu por assimilação, e estava estagnado,
                # nós damos a chance de resetar o delta para reexplorar o novo ponto
                if self.deltas[cluster_idx] < self.epsilon:
                    self.deltas[cluster_idx] = self.delta_start * 0.5 # Acorda mitigado

        # 2. Análise Estatística (Z-Score)
        media_votos = np.mean(votos)
        desvio_votos = np.std(votos) if np.std(votos) > 0 else 1e-6

        # 3. Decisão de Disparo de Busca Local Contínua
        for i in range(self.K):
            z_score = (votos[i] - media_votos) / desvio_votos

            # Critério de ativação: Alta densidade e NÃO estar estagnado no fundo do vale
            if z_score >= 1.645 and votos[i] > 0 and self.deltas[i] >= self.epsilon:

                # Armazena o fitness antes da busca para calcular a estatística de ganho
                fitness_anterior = self.ultimos_fitness[i]

                # DISPARA H&J CONTINUADO: Passa o delta em que parou na última vez
                novo_centro, novo_fitness, delta_final = hooke_jeeves_refinement(
                    engine, self.centros[i], delta_inicial=self.deltas[i], budget=budget_hj
                )

                # Atualiza o Estado do Cluster
                self.centros[i] = novo_centro
                self.ultimos_fitness[i] = novo_fitness
                self.deltas[i] = delta_final # Se exauriu o orçamento, delta_final veio grande. Se convergiu, veio < epsilon.

                # Estatística de Melhoria (Útil para o relatório/artigo)
                ganho = fitness_anterior - novo_fitness
                if ganho > 0:
                    print(f"Cluster {i} continuou descendo! Ganho nesta BL: {ganho:.2f}. Próximo Delta: {delta_final}")
                else:
                    # Se rodou a BL e o ganho foi irrelevante, força a estagnação para poupar CFOs
                    self.deltas[i] = self.epsilon * 0.9

        return self.centros

In [ ]:
import numpy as np

class ClusteringSearch:
    def __init__(self, num_clusters, dimensao, threshold_cooldown=3):
        self.K = num_clusters
        self.D = dimensao
        # Inicializa centros aleatórios no espaço [0, 1]
        self.centros = np.random.rand(num_clusters, dimensao)
        # Array para rastrear o Cooldown de cada cluster
        self.cooldown = np.zeros(num_clusters, dtype=int)
        self.threshold_cooldown = threshold_cooldown

    def executar_ciclo_geracao(self, novas_solucoes, fitness_novas_solucoes, engine,
                               usar_filtro_qualidade=False, z_limiar=1.645, budget_hj=500):
        """
        Executa o agrupamento e decide quais clusters disparam a busca local (H&J).
        """
        votos = np.zeros(self.K)
        # Lista para acumular os fitness das soluções que votaram em cada cluster
        fitness_por_cluster = {i: [] for i in range(self.K)}

        # 1. Cálculo do Raio Dinâmico da Literatura
        raio = 1.0 / (2 * (self.K ** (1 / self.D)))

        # 2. Fase de Agrupamento (Votação e Assimilação)
        for S, fit in zip(novas_solucoes, fitness_novas_solucoes):
            # Distância de Chebyshev (L_infinity)
            distancias = np.max(np.abs(self.centros - S), axis=1)
            cluster_mais_proximo = np.argmin(distancias)
            menor_distancia = distancias[cluster_mais_proximo]

            if menor_distancia <= raio:
                # Assimilação (Arrasto de 20%)
                self.centros[cluster_mais_proximo] = 0.8 * self.centros[cluster_mais_proximo] + 0.2 * S
                votos[cluster_mais_proximo] += 1
                fitness_por_cluster[cluster_mais_proximo].append(fit)
            else:
                # Outlier: Substitui o cluster que está há mais tempo sem atividade ou menos votado
                substituido_idx = np.argmin(votos)
                if self.cooldown[substituido_idx] == 0: # Só substitui se não estiver em cooldown
                    self.centros[substituido_idx] = S.copy()
                    votos[substituido_idx] = 1
                    fitness_por_cluster[substituido_idx] = [fit]

        # 3. Análise Estatística da Geração (Z-Score)
        media_votos = np.mean(votos)
        desvio_votos = np.std(votos)

        # Evita divisão por zero se todos os clusters receberem exatamente os mesmos votos
        if desvio_votos == 0:
            desvio_votos = 1e-6

        # Reduz o Cooldown dos clusters que estavam bloqueados
        self.cooldown = np.maximum(0, self.cooldown - 1)

        # 4. Julgamento dos Clusters Promissores
        for i in range(self.K):
            z_score = (votos[i] - media_votos) / desvio_votos

            # Condição 1: O cluster recebeu um volume significativo de votos?
            condicao_disparo = (z_score >= z_limiar) and (votos[i] > 0)

            # Condição 2 (Opcional): O cluster é de alta qualidade?
            if condicao_disparo and usar_filtro_qualidade:
                # Só dispara se a média do fitness desse cluster for MELHOR que a média global da geração
                media_fit_cluster = np.mean(fitness_por_cluster[i])
                media_fit_global = np.mean(fitness_novas_solucoes)
                if media_fit_cluster > media_fit_global: # Lembrete: se for minimização, 'maior' significa pior fitness
                    condicao_disparo = False

            # Condição 3: O cluster está livre do Cooldown?
            if condicao_disparo and (self.cooldown[i] == 0):
                # >>> DISPARA A BUSCA LOCAL (Hooke & Jeeves) <<<
                print(f"--> Cluster {i} promissor! (Z-Score: {z_score:.2f}). Disparando H&J.")

                # O H&J refina a partir do CENTRO do cluster atualizado
                novo_centro, novo_fitness = hooke_jeeves_refinement(engine, self.centros[i], budget=budget_hj)

                # Atualiza o centro do cluster com o resultado refinado da busca local
                self.centros[i] = novo_centro

                # Ativa o Cooldown para este cluster não reativar imediatamente na próxima geração
                self.cooldown[i] = self.threshold_cooldown

        return self.centros

In [ ]:
import numpy as np
from enum import Enum
from typing import List, Optional, Tuple, Callable

class ClusterState(Enum):
    ACTIVE = 1      # Aceita novos membros e executa Busca Local quando elegível
    COOLDOWN = 2    # Em tempo de espera; rejeita buscas locais caras
    EXHAUSTED = 3   # Região esgotada/convergida; desativado até perturbação

class Cluster:
    def __init__(
        self,
        cluster_id: int,
        centroid: np.ndarray,
        fitness: float,
        r_min: float = 0.05,
        r_max: float = 0.25,
        base_cooldown: int = 5,
        gamma: float = 1.5,
        max_cooldown: int = 50
    ):
        self.id = cluster_id
        self.centroid = np.copy(centroid)
        self.best_fitness = fitness

        # Raio adaptativo
        self.r_min = r_min
        self.r_max = r_max
        self.radius = (r_min + r_max) / 2.0

        # Memória de estado e Cooldown
        self.state = ClusterState.ACTIVE
        self.stagnation_count = 0
        self.cooldown_ticks = 0
        self.base_cooldown = base_cooldown
        self.gamma = gamma
        self.max_cooldown = max_cooldown

        self.membros_count = 1

    def chebyshev_distance(self, individual: np.ndarray) -> float:
        """Calcula a distância L_infinito (Chebyshev) entre o indivíduo e o centroide."""
        return float(np.max(np.abs(self.centroid - individual)))

    def is_inside(self, individual: np.ndarray) -> bool:
        """Verifica se o indivíduo está dentro da vizinhança hipercúbica R."""
        return self.chebyshev_distance(individual) <= self.radius

    def tick_cooldown(self):
        """Decrementa o temporizador de cooldown a cada geração."""
        if self.state == ClusterState.COOLDOWN:
            self.cooldown_ticks -= 1
            if self.cooldown_ticks <= 0:
                self.state = ClusterState.ACTIVE

    def register_local_search_result(self, improved: bool, new_centroid: np.ndarray, new_fitness: float):
        """Atualiza a memória do cluster com base no resultado da busca local."""
        if improved and new_fitness < self.best_fitness:
            # Sucesso: Reseta estagnação, atualiza centroide e ajusta raio (foco fino)
            self.centroid = np.copy(new_centroid)
            self.best_fitness = new_fitness
            self.stagnation_count = 0
            self.state = ClusterState.ACTIVE
            self.cooldown_ticks = 0

            # Contração adaptativa do raio para refinamento local
            self.radius = max(self.r_min, self.radius * 0.9)
        else:
            # Falha: Incrementa estagnação e aciona Cooldown Geométrico
            self.stagnation_count += 1

            # Expansão adaptativa do raio para tentar capturar vizinhos mais distantes
            self.radius = min(self.r_max, self.radius * 1.1)

            # Cálculo do tempo de cooldown geométrico
            cd = int(self.base_cooldown * (self.gamma ** self.stagnation_count))
            self.cooldown_ticks = min(self.max_cooldown, cd)
            self.state = ClusterState.COOLDOWN


class StatefulClusteringSearch:
    def __init__(
        self,
        num_clusters_max: int = 5,
        r_min: float = 0.05,
        r_max: float = 0.25,
        base_cooldown: int = 5,
        gamma: float = 1.5,
        max_stagnation: int = 10
    ):
        self.num_clusters_max = num_clusters_max
        self.r_min = r_min
        self.r_max = r_max
        self.base_cooldown = base_cooldown
        self.gamma = gamma
        self.max_stagnation = max_stagnation
        self.clusters: List[Cluster] = []
        self._next_id = 0

    def update_states(self):
        """Avança o temporizador de cooldown de todos os clusters."""
        for c in self.clusters:
            c.tick_cooldown()

    def process_individual(
        self,
        individual: np.ndarray,
        fitness: float,
        local_search_fn: Callable[[np.ndarray, float], Tuple[np.ndarray, float]]
    ) -> Tuple[np.ndarray, float, bool]:
        """
        Processa um indivíduo promissor na estrutura de clustering:
        1. Encontra o cluster mais próximo via Distância de Chebyshev.
        2. Se estiver dentro do raio adaptativo, absorve e decide se faz Busca Local.
        3. Se estiver fora, cria um novo cluster se houver vaga.
        """
        if not self.clusters:
            # Cria o primeiro cluster
            new_cluster = Cluster(
                self._next_id, individual, fitness,
                self.r_min, self.r_max, self.base_cooldown, self.gamma
            )
            self._next_id += 1
            self.clusters.append(new_cluster)
            return individual, fitness, False

        # Encontra o cluster mais próximo usando Chebyshev
        distances = [c.chebyshev_distance(individual) for c in self.clusters]
        min_idx = int(np.argmin(distances))
        closest_cluster = self.clusters[min_idx]
        min_dist = distances[min_idx]

        # Verifica se pertence ao cluster mais próximo
        if min_dist <= closest_cluster.radius:
            closest_cluster.membros_count += 1

            # Só executa Busca Local se o cluster estiver ATIVO
            if closest_cluster.state == ClusterState.ACTIVE:
                # Executa a busca local refinada no candidato/centroide
                ls_ind, ls_fit = local_search_fn(individual, fitness)
                improved = ls_fit < fitness or ls_fit < closest_cluster.best_fitness

                # Registra resultado e atualiza dinâmica de Cooldown/Raio
                closest_cluster.register_local_search_result(improved, ls_ind, ls_fit)
                return ls_ind, ls_fit, True
            else:
                # Cluster em COOLDOWN: ignora a busca local cara para economizar avaliações
                return individual, fitness, False

        else:
            # Fora da vizinhança: Cria novo cluster se não atingiu o limite
            if len(self.clusters) < self.num_clusters_max:
                new_cluster = Cluster(
                    self._next_id, individual, fitness,
                    self.r_min, self.r_max, self.base_cooldown, self.gamma
                )
                self._next_id += 1
                self.clusters.append(new_cluster)

                # Executa busca local no centroide da nova região descoberta
                ls_ind, ls_fit = local_search_fn(individual, fitness)
                improved = ls_fit < fitness
                new_cluster.register_local_search_result(improved, ls_ind, ls_fit)
                return ls_ind, ls_fit, True

        return individual, fitness, False